# Modern Data Science in Python: Great Tables

In this session, we'll learn how to turn DataFrames into polished, publication-quality tables using **Great Tables**. We'll work with real data from Posit's DevRel I/O platform, which tracks open source project activity across GitHub, PyPI, CRAN, and more.

By the end, you'll know how to:

- build a `GT` table from a Polars DataFrame
- add titles, spanners, and source notes
- format numbers, apply data-driven color, and use nanoplots
- style individual cells with the `style` + `loc` system

## Load the data

We have three Parquet files from DevRel I/O:

- `projects.parquet`: metadata for tracked open source projects
- `metrics.parquet`: daily metric snapshots (downloads, forks, issues, etc.)
- `events.parquet`: individual GitHub events (issues opened, PRs merged, etc.)

In [ ]:
import polars as pl
import polars.selectors as cs
from datetime import date

projects = pl.read_parquet("data/projects.parquet")
metrics = pl.read_parquet("data/metrics.parquet")
events = pl.read_parquet("data/events.parquet")

## Prepare Table 1: Project Overview

Let's create a summary of Posit's key open source projects showing their name, description, language, total package downloads, and GitHub activity.

In [ ]:
# Filter to featured projects (those with R or Python tags)
featured = projects.filter(
    pl.col("tags").is_not_null()
    & pl.col("tags").list.first().is_in(["r", "python"])
)
featured_ids = featured["id"].to_list()

# Aggregate total downloads across all time (PyPI + CRAN)
downloads = (
    metrics
    .filter(
        (pl.col("project").is_in(featured_ids))
        & (pl.col("metric") == "daily_downloads")
    )
    .group_by("project")
    .agg(pl.col("value").sum().alias("total_downloads"))
)

# Aggregate GitHub activity (forks, issues opened, PRs merged)
gh_activity = (
    metrics
    .filter(
        (pl.col("project").is_in(featured_ids))
        & (pl.col("source") == "github")
        & (pl.col("metric").is_in(["daily_forks", "daily_issues_opened", "daily_prs_merged"]))
    )
    .group_by("project", "metric")
    .agg(pl.col("value").sum().alias("total"))
    .pivot(on="metric", values="total")
)

# Join everything together, revise column names, sort by total downloads, and add a color column for the language
project_summary = (
    featured
    .select("id", "name", "description", "tags")
    .join(downloads, left_on="id", right_on="project", how="left")
    .join(
        gh_activity.select("project", "daily_forks", "daily_issues_opened", "daily_prs_merged"),
        left_on="id",
        right_on="project",
        how="left",
    )
    .with_columns(
        pl.col("tags").list.first()
        .replace({"r": "R", "python": "Python"})
        .alias("language")
    )
    .with_columns(
        pl.when(pl.col("language") == "R").then(pl.lit("#2266B1"))
        .when(pl.col("language") == "Python").then(pl.lit("#1B7D3A"))
        .otherwise(pl.lit("#555555"))
        .alias("lang_color")
    )
    .drop("tags", "id")
    .sort("total_downloads", descending=True, nulls_last=True)
)

project_summary

## Table 1: A basic GT table

Let's start simple and just pass the DataFrame to `GT()` (and add a header).

In [ ]:
from great_tables import GT, md, html, style, loc, from_column, nanoplot_options

(
    GT(project_summary)
    .tab_header(
        title="Posit Open Source Projects",
        subtitle="Key projects tracked by DevRel I/O",
    )
)

## Table 1: Formatting and labeling

Raw numbers are harder to read than they ought to be. So let's:

- format large numbers with separators
- rename columns to human-friendly labels
- handle missing values gracefully
- hide the `lang_color` column (we'll use it for styling later)

In [ ]:
(
    GT(project_summary)

    # Add a title and subtitle to the table
    .tab_header(
        title="Posit Open Source Projects",
        subtitle="Key projects tracked by DevRel I/O",
    )

    # Use compact integer formatting on several columns
    .fmt_integer(columns=cs.starts_with("daily_") | cs.by_name("total_downloads"), compact=True)

    # Replace missing values with a dash
    .sub_missing(missing_text="—")

    # Ensure that labels are more presentable
    .cols_label(
        name="Project",
        description="Description",
        language="Lang",
        total_downloads="Downloads",
        daily_forks="Forks",
        daily_issues_opened="Issues",
        daily_prs_merged="PRs Merged",
    )

    # Hide the unneeded `lang_color` column
    .cols_hide("lang_color")
)

## Table 1: Grouping Rows, using spanners, styling, and using `data_color()`

Now let's make it more presentation-ready by:

- using row groups to create row groupings for R and Python projects
- grouping the GitHub-based columns under a **spanner**
- using each languages `color` value to style the text of the project names
- applying a color scale to the downloads column

In [ ]:
(
    GT(project_summary, groupname_col="language")

    # Add a title and subtitle
    .tab_header(
        title="Posit Open Source Projects",
        subtitle="Activity summary from DevRel I/O",
    )

    # Format numeric columns as compact integers
    .fmt_integer(columns=cs.starts_with("daily_") | cs.by_name("total_downloads"), compact=True)

    # Replace missing values with a dash
    .sub_missing(missing_text="—")

    # Set human-friendly column labels
    .cols_label(
        name="Project",
        description="Description",
        total_downloads="Downloads",
        daily_forks="Forks",
        daily_issues_opened="Issues",
        daily_prs_merged="PRs Merged",
    )

    # Hide the helper column used for styling
    .cols_hide("lang_color")

    # Group the GitHub columns under a spanner
    .tab_spanner(label="GitHub Activity", columns=["daily_forks", "daily_issues_opened", "daily_prs_merged"])

    # Style the project name with the language color
    .tab_style(
        style=style.text(color=from_column("lang_color"), weight="bold"),
        locations=loc.body(columns="name"),
    )

    # Apply a blue color scale to the downloads column
    .data_color(
        columns="total_downloads",
        palette=["#f7fbff", "#08306b"],
        domain=[0, 200_000_000],
        na_color="transparent",
    )

    # Add a source note at the bottom
    .tab_source_note(md("Data from **DevRel I/O** (Posit, PBC). Downloads include PyPI and CRAN."))

    # Set a fixed width for the description column
    .cols_width(description="280px")
)

## Prepare Table 2: Python Package Download Trends

For our second table, we'll focus on **Python packages** and show their monthly download trends in 2026 using **nanoplots**. These are very small inline charts that are rendered directly inside the table cells.

In [ ]:
python_packages = ["plotnine", "great-tables", "shiny-python", "chatlas", "pointblank-python", "orbital-python"]

# Get monthly downloads in 2026
monthly_downloads = (
    metrics
    .filter(
        (pl.col("project").is_in(python_packages))
        & (pl.col("metric") == "daily_downloads")
        & (pl.col("date") >= date(2026, 1, 1))
    )
    .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
    .group_by("project", "month")
    .agg(pl.col("value").sum().alias("monthly_downloads"))
    .sort("project", "month")
)

# Build per-project summary with trend list, total, and latest month vs prior
download_trends = (
    monthly_downloads
    .group_by("project")
    .agg(
        pl.col("monthly_downloads").alias("monthly_trend"),
        pl.col("monthly_downloads").sum().alias("total_2026"),
        pl.col("monthly_downloads").last().alias("latest_month"),
        pl.col("monthly_downloads").mean().alias("avg_monthly"),
    )
    .sort("total_2026", descending=True)
)

# Join with project metadata for display names
download_trends = (
    download_trends
    .join(
        projects.select("id", "name"),
        left_on="project",
        right_on="id",
        how="left",
    )
    .select("name", "monthly_trend", "total_2026", "avg_monthly", "latest_month")
)

download_trends

## Table 2: Nanoplots and polished formatting

This table demonstrates:

- `fmt_nanoplot()` for inline sparkline charts
- `fmt_number()` with `compact=True` and `n_sigfig=3` for readable large numbers
- `opt_stylize()` for a quick theme

In [ ]:
(
    GT(download_trends)

    # Add a title and subtitle
    .tab_header(
        title="Python Package Downloads in 2026",
        subtitle="Monthly download trends from PyPI (Jan–Aug 2026)",
    )

    # Render bar-type nanoplots from the monthly trend list
    .fmt_nanoplot(
        "monthly_trend",
        plot_type="bar",
    )

    # Format numeric columns with 3 significant digits, compact
    .fmt_number(["total_2026", "latest_month", "avg_monthly"], n_sigfig=3, compact=True)

    # Set human-friendly column labels
    .cols_label(
        name="Package",
        monthly_trend="Monthly Trend",
        total_2026="Total (2026)",
        avg_monthly="Avg/Month",
        latest_month="Latest Month",
    )

    # Add a source note
    .tab_source_note("Source: PyPI download statistics via DevRel I/O")

    # Set a fixed width for the nanoplot column
    .cols_width(monthly_trend="200px")

    # Apply a predefined theme
    .opt_stylize(style=1)
)

## Table 2 (a variation): Line nanoplots with reference line

Let's use the same data, but instead make line-type nanoplots having a reference line showing the per-package average.

In [ ]:
(
    GT(download_trends)

    # Add a title and subtitle
    .tab_header(
        title="Python Package Downloads in 2026",
        subtitle="Monthly download trends from PyPI (Jan–Aug 2026)",
    )

    # Render line-type nanoplots with a mean reference line
    .fmt_nanoplot(
        "monthly_trend",
        plot_type="line",
        reference_line="mean",
    )

    # Format numeric columns with 3 significant digits, compact
    .fmt_number(["total_2026", "latest_month", "avg_monthly"], n_sigfig=3, compact=True)

    # Set human-friendly column labels
    .cols_label(
        name="Package",
        monthly_trend="Monthly Trend",
        total_2026="Total (2026)",
        avg_monthly="Avg/Month",
        latest_month="Latest Month",
    )

    # Add a source note
    .tab_source_note("Source: PyPI download statistics via DevRel I/O")

    # Set a fixed width for the nanoplot column
    .cols_width(monthly_trend="200px")

    # Apply a predefined theme
    .opt_stylize(style=1)
)

## Saving a table as PNG

You can use `.gtsave()` to export any GT table to a PNG (or HTML) file. Note that interactive elements like nanoplots will be rendered as static images.

In [ ]:
(
    GT(download_trends)
    .tab_header(
        title="Python Package Downloads in 2026",
        subtitle="Monthly download trends from PyPI (Jan–Aug 2026)",
    )
    .fmt_nanoplot(
        "monthly_trend",
        plot_type="line",
        reference_line="mean",
    )
    .fmt_number(["total_2026", "latest_month", "avg_monthly"], n_sigfig=3, compact=True)
    .cols_label(
        name="Package",
        monthly_trend="Monthly Trend",
        total_2026="Total (2026)",
        avg_monthly="Avg/Month",
        latest_month="Latest Month",
    )
    .tab_source_note("Source: PyPI download statistics via DevRel I/O")
    .cols_width(monthly_trend="200px")
    .opt_stylize(style=1)

    # Also try `expand=` (border padding) and `zoom=` (higher values mean sharper text)
    .gtsave("python_downloads_2026.png")
)

## Exercise

Using the `events` data, create your own summary table. Here are a few ideas:

1. Pick 5–10 projects and count their total events by type (forks, issues opened, PRs merged, comments)
2. Build a `GT` table with:
   - a meaningful title and subtitle
   - formatted numbers
   - at least one spanner grouping related columns
   - a `data_color()` call on one numeric column

As a bonus, try to use `tab_style()` with `style.borders()` or `style.fill()` to highlight the row with the most activity.

In [ ]:
# Your code here